In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/drought-pred'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/drought-pred/validation_timeseries.csv
/kaggle/input/drought-pred/train_timeseries.csv
/kaggle/input/drought-pred/soil_data.csv
/kaggle/input/drought-pred/test_timeseries.csv


In [2]:
# %%
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Load Weather Time-Series Data
train_weather = pd.read_csv('/kaggle/input/drought-pred/train_timeseries.csv')
val_weather = pd.read_csv('/kaggle/input/drought-pred/validation_timeseries.csv')
test_weather = pd.read_csv('/kaggle/input/drought-pred/test_timeseries.csv')

# Load Soil Data
soil_data = pd.read_csv('/kaggle/input/drought-pred/soil_data.csv')

# Fill Missing Values
train_weather.interpolate(method='ffill', inplace=True)
val_weather.interpolate(method='ffill', inplace=True)
test_weather.interpolate(method='ffill', inplace=True)
soil_data.fillna(soil_data.mean(), inplace=True)

# Ensure target column exists
if 'score' not in train_weather.columns:
    print("Error: 'score' column not found in train_weather dataset!")
    print("Available columns:", train_weather.columns)
    exit()

# Select Relevant Features
weather_features = ['PRECTOT', 'PS', 'QV2M', 'T2M', 'T2MDEW', 'T2MWET',
                    'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'TS', 'WS10M', 'WS10M_MAX',
                    'WS10M_MIN', 'WS10M_RANGE', 'WS50M', 'WS50M_MAX', 'WS50M_MIN']

soil_features = ['fips', 'lat', 'lon', 'elevation', 'slope1', 'slope2', 'slope3', 'slope4', 'slope5', 'slope6']

# Prepare Input Features
X_train_weather = train_weather[weather_features]
X_val_weather = val_weather[weather_features]
X_test_weather = test_weather[weather_features]
X_soil = soil_data[soil_features]

# Target Variable
y_train = train_weather['score']
y_val = val_weather['score']
y_test = test_weather['score']

# Normalize Features
weather_scaler = StandardScaler()
X_train_weather = weather_scaler.fit_transform(X_train_weather)
X_val_weather = weather_scaler.transform(X_val_weather)
X_test_weather = weather_scaler.transform(X_test_weather)

soil_scaler = StandardScaler()
X_soil = soil_scaler.fit_transform(X_soil)

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_weather, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_weather, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_weather, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Handle NaNs in target variables
def replace_nan(tensor):
    if torch.isnan(tensor).sum() > 0:
        mean_value = torch.nan_to_num(torch.mean(tensor[~torch.isnan(tensor)]))
        tensor[torch.isnan(tensor)] = mean_value
    return tensor

y_train_tensor = replace_nan(y_train_tensor)
y_val_tensor = replace_nan(y_val_tensor)
y_test_tensor = replace_nan(y_test_tensor)

# Create Datasets & DataLoaders
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("NaN in y_train after fix:", torch.isnan(y_train_tensor).sum().item())
print("NaN in y_val after fix:", torch.isnan(y_val_tensor).sum().item())
print("NaN in y_test after fix:", torch.isnan(y_test_tensor).sum().item())

# %%
import torch.nn as nn
import torch.nn.functional as F

class DroughtLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super(DroughtLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)  # Regression output

    def forward(self, x):
        x, _ = self.lstm(x.unsqueeze(1))  # Fix LSTM input shape
        x = self.fc(x[:, -1, :])  # Take last LSTM output
        return x

# %%
import torch.optim as optim

def train_model(model, train_loader, val_loader, epochs=10, lr=0.0001):
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses, val_accuracies, val_f1_scores = [], [], [], []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_losses.append(train_loss / len(train_loader))

        # Validation
        model.eval()
        val_loss, correct = 0.0, 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val, y_val = X_val.to(device), y_val.to(device)
                outputs = model(X_val)
                loss = criterion(outputs, y_val)
                val_loss += loss.item()

                preds = (outputs > 0.5).cpu().numpy().astype(int)
                labels = (y_val > 0.5).cpu().numpy().astype(int)

                all_preds.extend(preds)
                all_labels.extend(labels)

        val_losses.append(val_loss / len(val_loader))
        val_f1_scores.append(f1_score(all_labels, all_preds, average="binary", zero_division=1))

        print(f"Epoch {epoch+1}: Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}, Val F1: {val_f1_scores[-1]:.4f}")

# Train Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DroughtLSTM(input_size=X_train_tensor.shape[1])
train_model(model, train_loader, val_loader, epochs=10)


<ipython-input-2-490f9c22db69>:20: FutureWarning: DataFrame.interpolate with method=ffill is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  train_weather.interpolate(method='ffill', inplace=True)
<ipython-input-2-490f9c22db69>:21: FutureWarning: DataFrame.interpolate with method=ffill is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  val_weather.interpolate(method='ffill', inplace=True)
<ipython-input-2-490f9c22db69>:22: FutureWarning: DataFrame.interpolate with method=ffill is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  test_weather.interpolate(method='ffill', inplace=True)


NaN in y_train after fix: 0
NaN in y_val after fix: 0
NaN in y_test after fix: 0
Epoch 1: Train Loss: 1.2258, Val Loss: 0.8259, Val F1: 0.5285
Epoch 2: Train Loss: 1.2051, Val Loss: 0.8087, Val F1: 0.5323
Epoch 3: Train Loss: 1.1978, Val Loss: 0.8188, Val F1: 0.5357
Epoch 4: Train Loss: 1.1925, Val Loss: 0.8240, Val F1: 0.5357
Epoch 5: Train Loss: 1.1877, Val Loss: 0.8079, Val F1: 0.5367
Epoch 6: Train Loss: 1.1834, Val Loss: 0.8020, Val F1: 0.5393
Epoch 7: Train Loss: 1.1798, Val Loss: 0.8026, Val F1: 0.5411
Epoch 8: Train Loss: 1.1765, Val Loss: 0.8129, Val F1: 0.5388
Epoch 9: Train Loss: 1.1737, Val Loss: 0.8261, Val F1: 0.5387
Epoch 10: Train Loss: 1.1711, Val Loss: 0.8271, Val F1: 0.5361
